[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/logicmojo-ai-july-2026/blob/main/modules/module-4-llms-genai/09-fine-tuning-decision-and-data/micro-assignment/solution/solution.ipynb)

# Micro-assignment 4.9 (solution): a house-style SFT dataset for movies

Prepare a small supervised fine-tuning dataset that teaches a **house style** on
the movie track: answer a plot description with the film as `Title (Year)` plus one
spoiler-free line. This teaches a behavior/format, not facts. Then justify why
fine-tuning (not RAG or prompting) fits.

## 1. The house-style rule

In [ ]:
SYSTEM = (
    'You are a film guide. Given a plot description, reply with the film as '
    'Title (Year), then one spoiler-free sentence. If it is not a film you '
    'know, reply exactly: "I do not recognize that film."'
)

## 2. Build chat-format rows from (description, target) pairs

In [ ]:
PAIRS = [
    ("Hunting artificial humans in a rainy future city.", "Blade Runner (1982). A weary detective tracks synthetic humans and questions what makes us human."),
    ("A boy journeys to the land of the dead to find family.", "Coco (2017). A music-loving boy crosses into the afterlife and reconnects with his ancestors."),
    ("Scientists revive dinosaurs for a park.", "Jurassic Park (1993). A dinosaur theme park goes badly wrong when the attractions escape."),
    ("Thieves enter dreams to plant an idea.", "Inception (2010). A team of dream-thieves attempts to plant an idea deep in a target's mind."),
    ("A wrongly jailed banker keeps hope over long years.", "The Shawshank Redemption (1994). A banker endures prison while quietly holding on to hope."),
    ("A crew robs three casinos in one night.", "Ocean's Eleven (2001). A slick crew pulls off a simultaneous heist across three casinos."),
    ("A hobbit destroys a magic ring.", "I do not recognize that film."),
]

def to_rows(pairs):
    return [{"messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": d},
        {"role": "assistant", "content": t},
    ]} for d, t in pairs]

rows = to_rows(PAIRS)
print(len(rows), "examples")

## 3. Inspect one formatted example

In [ ]:
for m in rows[0]['messages']:
    print(f"[{m['role']}] {m['content']}")

## 4. Train/validation split

In [ ]:
import random
random.seed(42)
shuffled = rows[:]; random.shuffle(shuffled)
cut = max(1, int(0.8 * len(shuffled)))
train, val = shuffled[:cut], shuffled[cut:]
print("train:", len(train), "| val:", len(val))

## 5. Justification and one risk (reasoning)

**Why fine-tune, not RAG or prompt.** The goal is a *consistent format* (always
`Title (Year)` plus one spoiler-free line, and a fixed refusal). That is a behavior,
not missing knowledge, so RAG does not help, and a prompt can describe the format
but the model drifts across many calls. A few dozen consistent examples make the
style the default. If the goal were instead *which films exist* or their facts, that
would be RAG, not fine-tuning.

**One risk.** Keep the fine-tuning set disjoint from any evaluation set, or the
score is optimistic (overfitting). Also keep facts out of the target answers; teach
only the style, so the model does not memorize stale plot details.